In [ ]:
# Call RIPE stat API to get the first time seen and the last time seen by a route collector.
# This code is for prefix re-origination case.

import requests
from datetime import datetime, timedelta
import pandas as pd

asn_to_match = 198949

df = pd.read_csv("../data/prefix re-origination/as"+str(asn_to_match)+"_may.csv")



prefixes = df["Prefix"]
start_times = []
end_times = [] 

# Loop over each row instead of unique prefixes
for idx, row in df.iterrows():
    prefix = row["Prefix"]
    start_time = row["Date"]  # This is the corresponding date for this row
    
    dt = datetime.strptime(start_time, '%Y-%m-%d')
    start_time = str(start_time) + "T00:00:00"
    
    # Add one day 
    next_day = dt.replace(hour=23, minute=59, second=59)
    # Convert back to string in the same format
    end_time = next_day.isoformat()


    # Construct API URL
    url = (
        "https://stat.ripe.net/data/bgp-updates/data.json"
        f"?resource={prefix}&starttime={start_time}&endtime={end_time}"
    )

    # Fetch data
    response = requests.get(url)
    data = response.json()

    # Extract relevant BGP update entries
    timestamps = []

    for record in data.get("data", {}).get("updates", []):
        rec_type = record["type"]
        if rec_type == "A":
            path = record["attrs"]["path"]
            if path[-1] == asn_to_match:
                timestamps.append(record["timestamp"])

    # Report results
    if timestamps:
        # Convert strings to datetime objects
        dt_objects = [datetime.fromisoformat(ts) for ts in timestamps]

        # Find min and max
#         first_seen = min(dt_objects)
#         last_seen = max(dt_objects)
        
        first_seen = timestamps[0]
        last_seen = timestamps[-1]
        
        start_times.append(first_seen)
        end_times.append(last_seen)
        print(f"For prefix {prefix} seen from {first_seen} to {last_seen} UTC")
    else:
        print(f"For prefix {prefix} URL {url} was not seen at the end of any path.")

# Add StartTime and EndTime as new columns
df["StartTime"] = start_times
df["EndTime"] = end_times

# Save back to the same CSV
df.to_csv("../data/prefix re-homing/as"+str(asn_to_match)+"_may_details.csv", index=False)
print("Completed.")

In [51]:
# Call RIPE stat API to get the first time seen and the last time seen by a route collector.
# This code is for prefix re-homing case.

import requests
from datetime import datetime, timedelta
import pandas as pd
import time

# Retry helper
def safe_fetch(url, max_retries=5, backoff=2):
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, timeout=60)
            resp.raise_for_status()
            return resp.json()
        except (requests.exceptions.ChunkedEncodingError,
                requests.exceptions.ConnectionError,
                requests.exceptions.Timeout) as e:
            wait = backoff ** attempt
            print(f"Fetch error: {e}. Retrying in {wait}s... ({attempt+1}/{max_retries})")
            time.sleep(wait)
        except requests.exceptions.HTTPError as e:
            print(f"HTTP error {e.response.status_code}: {e}")
            break
    return {}  # Return empty if all retries fail

asn_to_match = 13335

df = pd.read_csv("../data/prefix re-homing/as"+str(asn_to_match)+"_may.csv")

prefixes = df["Prefix"]
start_times = []
end_times = []
origins = []
dates = []
scrubbing_confirmed_prefixes = []

# Loop over each row instead of unique prefixes
for idx, row in df.iterrows():
    prefix = row["Prefix"]
    origin = row["Origin"]
    date = row["Date"]
    
    start_time = row["Date"]  # This is the corresponding date for this row
    
    dt = datetime.strptime(start_time, '%Y-%m-%d')
    start_time = str(start_time) + "T00:00:00"
    
    # Add one day 
    next_day = dt.replace(hour=23, minute=59, second=59)
    # Convert back to string in the same format
    end_time = next_day.isoformat()


    # Construct API URL
    url = (
        "https://stat.ripe.net/data/bgp-updates/data.json"
        f"?resource={prefix}&starttime={start_time}&endtime={end_time}"
    )

    # Fetch data safely
    data = safe_fetch(url)

    # Extract relevant BGP update entries
    timestamps = []

    for record in data.get("data", {}).get("updates", []):
        rec_type = record["type"]
        if rec_type == "A":
            path = record["attrs"]["path"]
            if len(path) > 1 and path[-2] == asn_to_match:
                timestamps.append(record["timestamp"])

    # Report results
    if timestamps and len(timestamps) > 1:
        # Convert strings to datetime objects
        dt_objects = [datetime.fromisoformat(ts) for ts in timestamps]

        # Find min and max
#         first_seen = min(dt_objects)
#         last_seen = max(dt_objects)
#         print(f"For prefix {prefix}")
        first_seen = timestamps[0]
        last_seen = timestamps[-2]
        
        start_times.append(first_seen)
        end_times.append(last_seen)
        
        # Some scrubbing prefixes were seen only once, with single timestamp which do not confirm scrubbing.
        scrubbing_confirmed_prefixes.append(prefix)
        
        origins.append(origin)
        dates.append(date)
        
#         print(f"For prefix {prefix} seen from {first_seen} to {last_seen} UTC")
    else:
        print(f"Prefix: {prefix} Date: {row['Date']} was not seen at the end of any path.")

# Create a new dataframe with columns Date, Prefix, Origin, StartTime and EndTime as new columns
# Create DataFrame
df = pd.DataFrame({
    "Date": dates,
    "Prefix": scrubbing_confirmed_prefixes,
    "Origin": origins,
    "StartTime": start_times,
    "EndTime": end_times
})

# Save back to the same CSV
df.to_csv("../data/prefix re-homing/as"+str(asn_to_match)+"_may_details.csv", index=False)
print("Completed.")

Prefix: 2407:7580:2022::/48 Date: 2025-05-03 was not seen at the end of any path.
Prefix: 2407:7580:2022::/48 Date: 2025-05-05 was not seen at the end of any path.
Prefix: 2407:7580:2022::/48 Date: 2025-05-06 was not seen at the end of any path.
Prefix: 2001:44b8:206f::/48 Date: 2025-05-07 was not seen at the end of any path.
Prefix: 2001:44b8:407b::/48 Date: 2025-05-07 was not seen at the end of any path.
Prefix: 2001:44b8:5034::/48 Date: 2025-05-07 was not seen at the end of any path.
Prefix: 2407:7580:2022::/48 Date: 2025-05-08 was not seen at the end of any path.
Prefix: 170.245.190.0/24 Date: 2025-05-10 was not seen at the end of any path.
Prefix: 2600:14a0:f0::/48 Date: 2025-05-13 was not seen at the end of any path.
Prefix: 2405:4200:ffee::/48 Date: 2025-05-13 was not seen at the end of any path.
Prefix: 2406:c500:fffd::/48 Date: 2025-05-14 was not seen at the end of any path.
Prefix: 2407:7580:2022::/48 Date: 2025-05-14 was not seen at the end of any path.
Prefix: 2001:df2:2c00

In [13]:
# This code is for prefix re-homing case. Here, I had to reduce the date field of csv file by one day back. 
# The origin file is store in Aruba.
import requests
from datetime import datetime, timedelta
import pandas as pd

asn_to_match = 198949

df = pd.read_csv("../data/prefix re-homing/as"+str(asn_to_match)+"_may.csv")

# Convert 'date' column to datetime explicitly
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")

# Subtract one day
df["Date"] = df["Date"] - pd.Timedelta(days=1)

# Save back to CSV (optional)
df.to_csv("../data/prefix re-homing/as"+str(asn_to_match)+"_may.csv", index=False)

In [32]:
# Insert duration field in as<scrubber>_may_details.csv for prefix re-origination cases

import pandas as pd
scrubber = "198949"

df = pd.read_csv("../data/prefix re-origination/as"+scrubber+"_may_details.csv")

df['StartTime'] = pd.to_datetime(df['StartTime'], errors='coerce')
df['EndTime'] = pd.to_datetime(df['EndTime'], errors='coerce')
# Subtracting two datetime columns results in a timedelta object.
df['duration'] = df['EndTime'] - df['StartTime']
# For example, to get the total number of seconds:
df['DurationSeconds'] = df['duration'].dt.total_seconds()
# Or in minutes:
df['DurationMinutes'] = df['DurationSeconds'] / 60
# Or in a nicely formatted string:
# df['duration_formatted'] = df['duration'].astype(str).str.replace('0 days ', '', regex=False)

# Specify the columns you want to save
columns_to_save = ['Date', 'Prefix', 'StartTime', 'EndTime', 'DurationMinutes', 'DurationSeconds']

# Save only the specified columns to a CSV file
df.to_csv('../data/prefix re-origination/as'+scrubber+'_may_duration.csv', columns=columns_to_save, index=False)
print("Completed.")

Completed.


In [37]:
# Insert duration field in as<scrubber>_may.csv for prefix re-homing cases

import pandas as pd
scrubber = "13335"

df = pd.read_csv("../data/prefix re-homing/as"+scrubber+"_may_details.csv")

df['StartTime'] = pd.to_datetime(df['StartTime'], errors='coerce')
df['EndTime'] = pd.to_datetime(df['EndTime'], errors='coerce')
# Subtracting two datetime columns results in a timedelta object.
df['duration'] = df['EndTime'] - df['StartTime']
# For example, to get the total number of seconds:
df['DurationSeconds'] = df['duration'].dt.total_seconds()
# Or in minutes:
df['DurationMinutes'] = df['DurationSeconds'] / 60
# Or in a nicely formatted string:
# df['duration_formatted'] = df['duration'].astype(str).str.replace('0 days ', '', regex=False)

# Specify the columns you want to save
columns_to_save = ['Date', 'Prefix', 'StartTime', 'EndTime', 'DurationMinutes', 'DurationSeconds']

# Save only the specified columns to a CSV file
df.to_csv('../data/prefix re-homing/as'+scrubber+'_may_duration.csv', columns=columns_to_save, index=False)
print("Completed.")

Completed.


In [61]:
# Get count of prefixes repeatededly attacked
import pandas as pd

def find_and_sort_by_count(df, column_name):
    """
    Finds and sorts duplicate rows in a DataFrame based on the count of duplication.

    Args:
        df (pd.DataFrame): The input DataFrame.
        column_name (str): The name of the column to check for duplicates.

    Returns:
        pd.DataFrame: A new DataFrame with duplicate rows, sorted by the
                      count of duplication in descending order.
    """
    # 1. Get the counts of all values in the specified column
    counts = df[column_name].value_counts()
    
    # 2. Find all rows that are duplicates (keep=False marks all occurrences)
    duplicate_rows = df[df.duplicated(subset=[column_name], keep=False)].copy()
    
    # 3. Add a new column with the duplication count for each row
    duplicate_rows['DuplicationCount'] = duplicate_rows[column_name].map(counts)
    
    # 4. Sort the DataFrame by the new count column and then by the ID itself
    sorted_df = duplicate_rows.sort_values(
        by=['DuplicationCount', column_name], 
        ascending=[False, True]
    )
    
    # Optional: drop the temporary count column if you don't need it
    # sorted_df = sorted_df.drop(columns='DuplicationCount')
    
    return sorted_df

df = pd.read_csv("../data/merged_scrubbers_duration.csv")

# Use the function to get the sorted duplicate rows
result = find_and_sort_by_count(df, 'prefix')

# Specify the three columns you want to print
columns_to_print = ["Date", "prefix", "DuplicationCount", "DurationMinutes", "DurationSeconds"]

# Select and print the DataFrame with only those columns
selected_df = result[columns_to_print]
print(selected_df)


# print(result["Date"])

           Date               prefix  DuplicationCount  DurationMinutes  \
677  2025-05-01  2401:5a00:b000::/48                30       345.583333   
684  2025-05-02  2401:5a00:b000::/48                30       173.850000   
689  2025-05-03  2401:5a00:b000::/48                30        82.400000   
694  2025-05-04  2401:5a00:b000::/48                30       455.750000   
697  2025-05-05  2401:5a00:b000::/48                30       786.350000   
..          ...                  ...               ...              ...   
957  2025-05-28      91.197.216.0/24                 2      1604.816667   
121  2025-05-16      91.197.217.0/24                 2       823.733333   
937  2025-05-16      91.197.217.0/24                 2      2130.733333   
139  2025-05-22      91.197.218.0/24                 2       582.700000   
953  2025-05-22      91.197.218.0/24                 2      1925.866667   

     DurationSeconds  
677          20735.0  
684          10431.0  
689           4944.0  
694    

In [77]:
selected_df = selected_df.sort_values('DuplicationCount', ascending=False)
selected_df.iloc[10:70]

,Date,prefix,DuplicationCount,DurationMinutes,DurationSeconds
697,2025-05-05,2401:5a00:b000::/48,30,786.350000,47181.0
806,2025-05-18,2401:5a00:b000::/48,30,0.000000,0.0
866,2025-05-26,2401:5a00:b000::/48,30,0.016667,1.0
684,2025-05-02,2401:5a00:b000::/48,30,173.850000,10431.0
763,2025-05-12,2401:5a00:b000::/48,30,115.250000,6915.0
888,2025-05-30,2401:5a00:b000::/48,30,1150.366667,69022.0
694,2025-05-04,2401:5a00:b000::/48,30,455.750000,27345.0
829,2025-05-21,2401:5a00:b000::/48,30,599.216667,35953.0
800,2025-05-17,2401:5a00:b000::/48,30,0.000000,0.0
753,2025-05-11,2401:5a00:b000::/48,30,480.650000,28839.0


In [41]:
# Merge _duration files of each scrubber from prefix re-origination case and prefix re-homing case
import os
import pandas as pd

dir_a = "../data/prefix re-origination/"
dir_b = "../data/prefix re-homing/"
output_dir = "../data/merged"

os.makedirs(output_dir, exist_ok=True)

# Get common files between the two dirs
files_a = set(os.listdir(dir_a))
files_b = set(os.listdir(dir_b))

# Only keep files ending with "_duration.csv"
common_files = {f for f in files_a.intersection(files_b) if f.endswith("_duration.csv")}

for fname in common_files:
    # Read CSVs
    df_a = pd.read_csv(os.path.join(dir_a, fname))
    df_b = pd.read_csv(os.path.join(dir_b, fname))
    
    # Merge (row-wise append)
    merged = pd.concat([df_a, df_b], ignore_index=True)
    
    # Save merged file
    out_path = os.path.join(output_dir, fname)
    merged.to_csv(out_path, index=False)
    print(f"Merged {fname} → {out_path}")


Merged as19905_may_duration.csv → ../data/merged/as19905_may_duration.csv
Merged as13335_may_duration.csv → ../data/merged/as13335_may_duration.csv
Merged as19551_may_duration.csv → ../data/merged/as19551_may_duration.csv
Merged as32787_may_duration.csv → ../data/merged/as32787_may_duration.csv
Merged as198949_may_duration.csv → ../data/merged/as198949_may_duration.csv


In [42]:
# Merge _duration files of all the scrubbers into one file
import os
import pandas as pd
import glob
import os

output_dir = "../data/merged"


# Get all CSV files in the folder
csv_files = glob.glob(os.path.join(output_dir, "*.csv"))

# Read and merge all CSVs
df_list = [pd.read_csv(file) for file in csv_files]
merged_df = pd.concat(df_list, ignore_index=True)

# Save to a new CSV
merged_df.to_csv(output_dir + "/merged_scrubbers_duration.csv", index=False)

In [10]:
# For the case where scrubbing started before 00:00:00 UTC, where we compare RIBs on next and previous day.
# Call RIPE stat API to get the first time seen and the last time seen by a route collector.
# This code is for prefix re-origination case.
# Input file is from Aruba merge_after_check_origin_asn_asn_prev_day_step4_old.py named 
# as_19905_diff_may_reorigination.csv
# It has fields date, origin and prefix. date is the day where we suspect scrubbing. 

import requests
from datetime import datetime, timedelta
import pandas as pd

asn_to_match = 19905
results = []

df = pd.read_csv("../data/diff_ribs/prefix re_origination/as_"+str(asn_to_match)+"_diff_may_reorigination.csv")

for idx, row in df.iterrows():
    prefix = row["prefix"]
    cur_time = row["date"]  # format YYYY-MM-DD
    origin = row["origin"]

    # Parse date
    start_dt = datetime.strptime(cur_time, "%Y-%m-%d")

    # Start time (previous day at 00:00:01)
    start_time = (start_dt - timedelta(days=1)).replace(hour=0, minute=0, second=1)
    start_time_str = start_time.strftime("%Y-%m-%dT%H:%M:%S")

    # End time (next day at 23:59:59)
    end_time = start_dt.replace(hour=23, minute=59, second=59)
    end_time_str = end_time.strftime("%Y-%m-%dT%H:%M:%S")

    # Construct API URL
    url = (
        "https://stat.ripe.net/data/bgp-updates/data.json"
        f"?resource={prefix}&starttime={start_time_str}&endtime={end_time_str}"
    )

    # Fetch data
    response = requests.get(url)
    data = response.json()

    # Extract relevant BGP update entries
    timestamps = []
    for record in data.get("data", {}).get("updates", []):
        if record["type"] == "A":
            path = record["attrs"]["path"]
            if path[-1] == asn_to_match:
                timestamps.append(record["timestamp"])

    # Store results only if seen
    if timestamps:
        first_seen = timestamps[0]
        last_seen = timestamps[-1]

        results.append({
            "prefix": prefix,
            "origin": origin,
            "StartTime": first_seen,
            "EndTime": last_seen
        })

        print(f"For prefix {prefix} seen from {first_seen} to {last_seen} UTC")
    else:
        print(f"For prefix {prefix}, URL {url} was not seen at the end of any path.")

# Convert to DataFrame and save to CSV
seen_df = pd.DataFrame(results)

# Save back to the same CSV
seen_df.to_csv("../data/diff_ribs/prefix re-origination/as"+str(asn_to_match)+"_diff_may_reorigination_details.csv", index=False)
print("Completed.")

For prefix 46.184.88.0/24 seen from 2025-05-02T03:19:13 to 2025-05-03T23:59:35 UTC
For prefix 78.41.60.0/24 seen from 2025-05-06T03:31:25 to 2025-05-07T23:08:34 UTC
For prefix 168.159.212.0/24 seen from 2025-05-08T22:53:02 to 2025-05-08T23:59:08 UTC
For prefix 188.248.77.0/24 seen from 2025-05-08T17:12:02 to 2025-05-09T22:54:55 UTC
For prefix 46.184.88.0/24 seen from 2025-05-17T17:32:02 to 2025-05-17T23:40:08 UTC
For prefix 200.27.97.0/24 seen from 2025-05-18T19:36:03 to 2025-05-18T19:44:13 UTC
For prefix 200.27.98.0/24 seen from 2025-05-18T19:36:03 to 2025-05-18T19:44:13 UTC
For prefix 45.169.54.0/24 seen from 2025-05-18T19:36:02 to 2025-05-18T23:58:09 UTC
For prefix 45.169.55.0/24 seen from 2025-05-18T19:36:02 to 2025-05-18T23:58:09 UTC
For prefix 109.230.113.0/24 seen from 2025-05-18T00:31:06 to 2025-05-19T23:46:58 UTC
For prefix 194.117.63.0/24 seen from 2025-05-18T00:31:06 to 2025-05-19T23:31:50 UTC
For prefix 78.41.60.0/24 seen from 2025-05-18T01:04:37 to 2025-05-19T23:31:42 UTC


In [43]:
# Insert duration field in as<scrubber>_may_details.csv for prefix re-homing cases for scrubbing started before 00:00:00 UTC.

import pandas as pd
scrubber = "198949"

df = pd.read_csv("../data/diff_ribs/prefix re_homing/as"+scrubber+"_diff_may_rehoming_details.csv")

df['StartTime'] = pd.to_datetime(df['StartTime'], errors='coerce')
df['EndTime'] = pd.to_datetime(df['EndTime'], errors='coerce')
# Subtracting two datetime columns results in a timedelta object.
df['duration'] = df['EndTime'] - df['StartTime']
# For example, to get the total number of seconds:
df['DurationSeconds'] = df['duration'].dt.total_seconds()
# Or in minutes:
df['DurationMinutes'] = df['DurationSeconds'] / 60
# Or in a nicely formatted string:
# df['duration_formatted'] = df['duration'].astype(str).str.replace('0 days ', '', regex=False)

# Specify the columns you want to save
columns_to_save = ['prefix', 'StartTime', 'EndTime', 'DurationMinutes', 'DurationSeconds']

# Save only the specified columns to a CSV file
df.to_csv('../data/diff_ribs/prefix re_homing/as'+scrubber+'_diff_may_rehoming_details_duration.csv', columns=columns_to_save, index=False)
print("Completed.")

Completed.


In [37]:
# Insert duration field in as<scrubber>_may_details.csv for prefix re-origination cases for scrubbing started before 00:00:00 UTC.

import pandas as pd
scrubber = "19905"

df = pd.read_csv("../data/diff_ribs/prefix re_origination/as"+scrubber+"_diff_may_reorigination_details.csv")

df['StartTime'] = pd.to_datetime(df['StartTime'], errors='coerce')
df['EndTime'] = pd.to_datetime(df['EndTime'], errors='coerce')
# Subtracting two datetime columns results in a timedelta object.
df['duration'] = df['EndTime'] - df['StartTime']
# For example, to get the total number of seconds:
df['DurationSeconds'] = df['duration'].dt.total_seconds()
# Or in minutes:
df['DurationMinutes'] = df['DurationSeconds'] / 60
# Or in a nicely formatted string:
# df['duration_formatted'] = df['duration'].astype(str).str.replace('0 days ', '', regex=False)

# Specify the columns you want to save
columns_to_save = ['prefix', 'origin', 'StartTime', 'EndTime', 'DurationMinutes', 'DurationSeconds']

# Save only the specified columns to a CSV file
df.to_csv('../data/diff_ribs/prefix re_origination/as'+scrubber+'_diff_may_reorigination_details_duration.csv', columns=columns_to_save, index=False)
print("Completed.")

Completed.


In [10]:
# For the case where scrubbing started before 00:00:00 UTC, where we compare RIBs on next and previous day.
# Call RIPE stat API to get the first time seen and the last time seen by a route collector.
# This code is for prefix re-homing case.
# Input file is from Aruba merge_after_check_origin_asn_asn_prev_day_step4_old.py named 
# as_19905_diff_may_rehoming.csv
# It has fields date, origin and prefix. date is the day where we suspect scrubbing. 

import requests
from datetime import datetime, timedelta
import pandas as pd

asn_to_match = 13335
results = []

df = pd.read_csv("../data/diff_ribs/prefix re_homing/as_"+str(asn_to_match)+"_diff_may_rehoming.csv")

for idx, row in df.iterrows():
    prefix = row["prefix"]
    cur_time = row["date"]  # format YYYY-MM-DD

    # Parse date
    start_dt = datetime.strptime(cur_time, "%Y-%m-%d")

    # Start time (previous day at 00:00:01)
    start_time = (start_dt - timedelta(days=1)).replace(hour=0, minute=0, second=1)
    start_time_str = start_time.strftime("%Y-%m-%dT%H:%M:%S")

    # End time (next day at 23:59:59)
    end_time = start_dt.replace(hour=23, minute=59, second=59)
    end_time_str = end_time.strftime("%Y-%m-%dT%H:%M:%S")

    # Construct API URL
    url = (
        "https://stat.ripe.net/data/bgp-updates/data.json"
        f"?resource={prefix}&starttime={start_time_str}&endtime={end_time_str}"
    )

    # Fetch data
    response = requests.get(url)
    data = response.json()

    # Extract relevant BGP update entries
    timestamps = []
    for record in data.get("data", {}).get("updates", []):
        if record["type"] == "A":
            path = record["attrs"]["path"]
            if len(path) > 1 and path[-2] == asn_to_match:
                timestamps.append(record["timestamp"])

    # Store results only if seen
    if timestamps:
        first_seen = timestamps[0]
        last_seen = timestamps[-1]

        results.append({
            "prefix": prefix,
            "StartTime": first_seen,
            "EndTime": last_seen
        })

        print(f"For prefix {prefix} seen from {first_seen} to {last_seen} UTC")
    else:
        print(f"For prefix {prefix}, URL {url} was not seen at the end of any path.")

# Convert to DataFrame and save to CSV
seen_df = pd.DataFrame(results)

# Save back to the same CSV
seen_df.to_csv("../data/diff_ribs/prefix re_homing/as"+str(asn_to_match)+"_diff_may_rehoming_details.csv", index=False)
print("Completed.")

For prefix 109.234.162.0/24 seen from 2025-05-01T08:53:58 to 2025-05-02T21:07:56 UTC
For prefix 1.209.185.0/24 seen from 2025-05-06T22:53:03 to 2025-05-06T23:59:14 UTC
For prefix 203.246.187.0/24 seen from 2025-05-06T23:11:40 to 2025-05-06T23:59:14 UTC
For prefix 103.115.76.0/24, URL https://stat.ripe.net/data/bgp-updates/data.json?resource=103.115.76.0/24&starttime=2025-05-07T00:00:01&endtime=2025-05-08T23:59:59 was not seen at the end of any path.
For prefix 103.50.106.0/24, URL https://stat.ripe.net/data/bgp-updates/data.json?resource=103.50.106.0/24&starttime=2025-05-07T00:00:01&endtime=2025-05-08T23:59:59 was not seen at the end of any path.
For prefix 104.243.8.0/24, URL https://stat.ripe.net/data/bgp-updates/data.json?resource=104.243.8.0/24&starttime=2025-05-07T00:00:01&endtime=2025-05-08T23:59:59 was not seen at the end of any path.
For prefix 109.234.160.0/24, URL https://stat.ripe.net/data/bgp-updates/data.json?resource=109.234.160.0/24&starttime=2025-05-07T00:00:01&endtime=

For prefix 195.250.17.0/24, URL https://stat.ripe.net/data/bgp-updates/data.json?resource=195.250.17.0/24&starttime=2025-05-07T00:00:01&endtime=2025-05-08T23:59:59 was not seen at the end of any path.
For prefix 195.250.19.0/24, URL https://stat.ripe.net/data/bgp-updates/data.json?resource=195.250.19.0/24&starttime=2025-05-07T00:00:01&endtime=2025-05-08T23:59:59 was not seen at the end of any path.
For prefix 197.255.200.0/24, URL https://stat.ripe.net/data/bgp-updates/data.json?resource=197.255.200.0/24&starttime=2025-05-07T00:00:01&endtime=2025-05-08T23:59:59 was not seen at the end of any path.
For prefix 198.80.208.0/24, URL https://stat.ripe.net/data/bgp-updates/data.json?resource=198.80.208.0/24&starttime=2025-05-07T00:00:01&endtime=2025-05-08T23:59:59 was not seen at the end of any path.
For prefix 198.80.209.0/24, URL https://stat.ripe.net/data/bgp-updates/data.json?resource=198.80.209.0/24&starttime=2025-05-07T00:00:01&endtime=2025-05-08T23:59:59 was not seen at the end of any

For prefix 2407:7580:2022::/48 seen from 2025-05-21T23:04:23 to 2025-05-21T23:04:23 UTC
For prefix 109.234.161.0/24 seen from 2025-05-22T00:19:27 to 2025-05-23T23:15:50 UTC
For prefix 109.234.162.0/24 seen from 2025-05-22T00:36:41 to 2025-05-23T23:15:50 UTC
For prefix 109.234.163.0/24 seen from 2025-05-22T00:19:27 to 2025-05-23T23:15:50 UTC
For prefix 162.255.219.0/24 seen from 2025-05-24T23:07:56 to 2025-05-24T23:56:18 UTC
For prefix 66.220.58.0/24 seen from 2025-05-23T00:11:03 to 2025-05-24T23:19:57 UTC
For prefix 109.234.160.0/24 seen from 2025-05-24T00:08:24 to 2025-05-24T19:05:22 UTC
For prefix 185.150.40.0/24 seen from 2025-05-24T00:03:01 to 2025-05-25T23:59:52 UTC
For prefix 203.99.60.0/24 seen from 2025-05-24T00:04:57 to 2025-05-25T23:58:27 UTC
For prefix 2407:7580:2022::/48 seen from 2025-05-26T07:59:52 to 2025-05-27T21:45:45 UTC
For prefix 138.255.62.0/24 seen from 2025-05-27T23:30:46 to 2025-05-28T23:46:48 UTC
For prefix 192.133.12.0/24 seen from 2025-05-27T00:02:13 to 2025-

In [44]:
# Merge _duration files of each scrubber from prefix re-origination case and prefix re-homing case
# After checking scrubbing that occured before 00:00:00 UTC.
import os
import pandas as pd

# Paths to your folders
folder_a = "../data/diff_ribs/prefix re_origination/"
folder_b = "../data/diff_ribs/prefix re_homing/"
output_folder = "../data/merged_v2"
os.makedirs(output_folder, exist_ok=True)

# Helper: extract ASN from filename (after 'as' up to '_')
def extract_asn(filename):
    return filename.split("_")[0][2:]

# Collect files
files_a = {extract_asn(f): os.path.join(folder_a, f) for f in os.listdir(folder_a) if f.endswith("_details_duration.csv")}
files_b = {extract_asn(f): os.path.join(folder_b, f) for f in os.listdir(folder_b) if f.endswith("_details_duration.csv")}

# Find common ASNs
common_asns = set(files_a.keys()) & set(files_b.keys())

for asn in common_asns:
    df_a = pd.read_csv(files_a[asn])
    df_b = pd.read_csv(files_b[asn])
    
    # Merge: here I just concatenate; adjust if you want a join on columns
    merged = pd.concat([df_a, df_b], ignore_index=True)
    
    out_file = os.path.join(output_folder, f"as{asn}_merged.csv")
    merged.to_csv(out_file, index=False)
    print(f"Merged ASN {asn} -> {out_file}")


Merged ASN 198949 -> ../data/merged_v2/as198949_merged.csv
Merged ASN 19905 -> ../data/merged_v2/as19905_merged.csv
Merged ASN 13335 -> ../data/merged_v2/as13335_merged.csv
Merged ASN 19551 -> ../data/merged_v2/as19551_merged.csv
Merged ASN 32787 -> ../data/merged_v2/as32787_merged.csv


In [45]:
# Merge _duration files of all the scrubbers inside merged_v2 into one file 
import os
import pandas as pd
import glob
import os

output_dir = "../data/merged_v2"


# Get all CSV files in the folder
csv_files = glob.glob(os.path.join(output_dir, "*.csv"))

# Read and merge all CSVs
df_list = [pd.read_csv(file) for file in csv_files]
merged_df = pd.concat(df_list, ignore_index=True)

# Save to a new CSV
merged_df.to_csv(output_dir + "/merged_scrubbers_duration_v2.csv", index=False)

In [53]:
# merge merged_scrubbers_duration_v1.csv and merged_scrubbers_duration_v1.csv
# Merge _duration files of all the scrubbers inside merged_v2 into one file 
import os
import pandas as pd
import glob
import os

output_dir = "../data/"

df1 = pd.read_csv("../data/merged/merged_scrubbers_duration.csv")

# Assume df1 and df2 are loaded
df1 = df1.rename(columns={"Prefix": "prefix"})
# Append with union of columns


df2 = pd.read_csv("../data/merged_v2/merged_scrubbers_duration_v2.csv")
merged_df = pd.concat([df1, df2], ignore_index=True)

# Save to a new CSV
merged_df.to_csv(output_dir + "/merged_scrubbers_duration.csv", index=False)

In [58]:
# Some rows have empty Date column value, Take them from StartTime values
import pandas as pd

df = pd.read_csv("../data/merged_scrubbers_duration.csv")

# Parse StartTime as datetime
df["StartTime"] = pd.to_datetime(df["StartTime"])

# Fill missing Date values from StartTime
df["Date"] = df["Date"].fillna(df["StartTime"].dt.date.astype(str))
df.to_csv(output_dir + "../data/merged_scrubbers_duration.csv", index=False)